In [1]:
import pandas as pd 
import re
from pathlib import Path
from statistics import median



Dado nosso primeiro trabalho com a base de entender sua estrutura e limpar os dados gerando nossa camanda bronze . 

Outra questão permeia uma das aplicações que quero fazer um llm local com essa base de conhecimento especialista nas obras. 

Para isso irei fazer um RAG

Mas veja só o problema, algumas obras já tem a divisão semantica a qual chamamos de capitulos e outras não 

para realizar os chunks eu quero uma abordagem hibrida ja aproveitando parte da semantica do capitulo quando esta exigir

Definindo um  tamanho de chunk e.g 500 carecteres, e aplicar dentro do capitulo quando existir e quando não direto na obra. 


bora lá ver quais obras tem ou não capitulos e como vamos abordar isso 

In [ ]:
PADRAO_CAPITULO = re.compile(
    r"""
    ^[ \t]*
    cap[ií]tulo
    [ \t]+
    (?P<numero>
        \d{1,3}           # CAPÍTULO 1, CAPÍTULO 23
        |
        [ivxlcdm]+        # CAPÍTULO I, XII, CXL
        |
        primeiro          # caso de Dom Casmurro
    )
    (?:
        [ \t]*
        [.\-–—:]
        [ \t]*
        [^\n]{0,100}
    )?
    [ \t]*$
    """,
    flags=re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)


def check_obra_tem_capitulos(
    txt_path,
    min_capitulos=1,
    min_mediana_chars=200,
):
    txt_path = Path(txt_path)

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    matches = list(
        PADRAO_CAPITULO.finditer(texto)
    )

    capitulos = [
        {
            "cabecalho": match.group(0).strip(),
            "numero": match.group("numero"),
            "inicio": match.start(),
        }
        for match in matches
    ]

  
    distancias = [
        atual["inicio"] - anterior["inicio"]
        for anterior, atual
        in zip(capitulos, capitulos[1:])
    ]

    mediana_distancia = (
        median(distancias)
        if distancias
        else 0
    )

    tem_capitulos = (
        len(capitulos) >= min_capitulos
        and mediana_distancia >= min_mediana_chars
    )

    return {
        "tem_capitulos": tem_capitulos,
        "qtd_capitulos_detectados": len(capitulos),
        "mediana_chars_entre_capitulos": mediana_distancia,
        "capitulos": capitulos,
    }

In [3]:
def romano_para_int(romano):
    valores = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000,
    }

    total = 0
    anterior = 0

    for letra in romano.upper()[::-1]:
        valor = valores[letra]

        if valor < anterior:
            total -= valor
        else:
            total += valor
            anterior = valor

    return total


def numero_capitulo(valor):
    valor = str(valor).strip().lower()

    if valor == "primeiro":
        return 1

    if valor.isdigit():
        return int(valor)

    return romano_para_int(
        valor.upper()
    )

In [4]:
PROJECT_ROOT = Path("..")

dom_casmurro_path = (
    PROJECT_ROOT
    / "Dataset"
    / "bronze"
    / "Dom_Casmurro.txt"
)

resultado = check_obra_tem_capitulos(
    dom_casmurro_path
)

print(
    "Tem capítulos:",
    resultado["tem_capitulos"],
)

print(
    "Capítulos detectados:",
    resultado["qtd_capitulos_detectados"],
)

print(
    "Mediana de caracteres entre capítulos:",
    resultado["mediana_chars_entre_capitulos"],
)

print("\nPrimeiros capítulos encontrados:")

for capitulo in resultado["capitulos"]:
    print(
        capitulo["cabecalho"]
    )

Tem capítulos: True
Capítulos detectados: 145
Mediana de caracteres entre capítulos: 2275.0

Primeiros capítulos encontrados:
CAPÍTULO I
CAPÍTULO II
CAPÍTULO III
CAPÍTULO IV
CAPÍTULO V
CAPÍTULO VI
CAPÍTULO VII
CAPÍTULO VIII
CAPÍTULO IX
CAPÍTULO XI
CAPÍTULO XI
CAPÍTULO XII
CAPÍTULO XIII
CAPÍTULO XIV
CAPÍTULO XV
CAPÍTULO XVI
CAPÍTULO XVII
CAPÍTULO XVIII
CAPÍTULO XIX
CAPÍTULO XX
CAPÍTULO XXI
CAPÍTULO XXII
CAPÍTULO XXIII
CAPÍTULO XXIV
CAPÍTULO XXVI
CAPÍTULO XXVI
CAPÍTULO XXVII
CAPÍTULO XXVIII
CAPÍTULO XXIX
CAPÍTULO XXX
CAPÍTULO XXXI
CAPÍTULO XXXII
CAPÍTULO XXXIII
CAPÍTULO XXXIV
CAPÍTULO XXXV
CAPÍTULO XXXVI
CAPÍTULO XXXVII
CAPÍTULO XXXVIII
CAPÍTULO XXXIX
CAPÍTULO XL
CAPÍTULO XLI
CAPÍTULO XLII
CAPÍTULO XLIII
CAPÍTULO XLIV
CAPÍTULO XLV
CAPÍTULO XLVI
CAPÍTULO XLVII
CAPÍTULO XLVIII
CAPÍTULO XLIX
CAPÍTULO L
CAPÍTULO LI
CAPÍTULO LII
CAPÍTULO LIII
CAPÍTULO LIV
CAPÍTULO LV
CAPÍTULO LVI
CAPÍTULO LVII
CAPÍTULO LVIII
CAPÍTULO LIX
CAPÍTULO LX
CAPÍTULO LXI
CAPÍTULO LXII
CAPÍTULO LXIII
CAPÍTULO LXIV
CAPÍ

In [5]:
capitulos_ordenados = sorted(
    resultado["capitulos"],
    key=lambda x: numero_capitulo(
        x["numero"]
    ),
)

for capitulo in capitulos_ordenados:
    numero = numero_capitulo(
        capitulo["numero"]
    )

    print(
        f"{numero:03d} -> "
        f"{capitulo['cabecalho']}"
    )

001 -> CAPÍTULO I
002 -> CAPÍTULO II
003 -> CAPÍTULO III
004 -> CAPÍTULO IV
005 -> CAPÍTULO V
006 -> CAPÍTULO VI
007 -> CAPÍTULO VII
008 -> CAPÍTULO VIII
009 -> CAPÍTULO IX
011 -> CAPÍTULO XI
011 -> CAPÍTULO XI
012 -> CAPÍTULO XII
013 -> CAPÍTULO XIII
014 -> CAPÍTULO XIV
015 -> CAPÍTULO XV
016 -> CAPÍTULO XVI
017 -> CAPÍTULO XVII
018 -> CAPÍTULO XVIII
019 -> CAPÍTULO XIX
020 -> CAPÍTULO XX
021 -> CAPÍTULO XXI
022 -> CAPÍTULO XXII
023 -> CAPÍTULO XXIII
024 -> CAPÍTULO XXIV
026 -> CAPÍTULO XXVI
026 -> CAPÍTULO XXVI
027 -> CAPÍTULO XXVII
028 -> CAPÍTULO XXVIII
029 -> CAPÍTULO XXIX
030 -> CAPÍTULO XXX
031 -> CAPÍTULO XXXI
032 -> CAPÍTULO XXXII
033 -> CAPÍTULO XXXIII
034 -> CAPÍTULO XXXIV
035 -> CAPÍTULO XXXV
036 -> CAPÍTULO XXXVI
037 -> CAPÍTULO XXXVII
038 -> CAPÍTULO XXXVIII
039 -> CAPÍTULO XXXIX
040 -> CAPÍTULO XL
041 -> CAPÍTULO XLI
042 -> CAPÍTULO XLII
043 -> CAPÍTULO XLIII
044 -> CAPÍTULO XLIV
045 -> CAPÍTULO XLV
046 -> CAPÍTULO XLVI
047 -> CAPÍTULO XLVII
048 -> CAPÍTULO XLVIII
049 ->

In [6]:
from collections import Counter


numeros = [
    numero_capitulo(
        capitulo["numero"]
    )
    for capitulo in resultado["capitulos"]
]

contagem = Counter(
    numeros
)

duplicados = {
    numero: qtd
    for numero, qtd in contagem.items()
    if qtd > 1
}

print(
    "Duplicados:",
    duplicados,
)

Duplicados: {11: 2, 26: 2, 60: 2, 95: 2}


Dom casmurro na base tem capitulos repetidos no proprio pdf! Que barra hein, terei que decidir algo sobre isso , isso levanta duvidas da propria fonte, o nosso extrator retonou 145 capitulos, mas via numerla romando temos até o 148 
148 -> CAPÍTULO CXLVIII 


Dom casmurro em especifico tem titulo pra capitolos logo após as ancoras, mas não se se outras obras tem!

Memorias postumas e  O alienista também tem titulos de capitlos 


Para seguir, vou considera o extrator como correto e tentar puxar o nome do titulo também 

In [ ]:


PADRAO_CAPITULO = re.compile(
    r"""
    ^[ \t]*
    cap[ií]tulo
    [ \t]+
    (?P<numero>
        \d{1,3}
        |
        [ivxlcdm]+
        |
        primeiro
    )
    (?:
        [ \t]*
        [\-–—.:]
        [ \t]*
        (?P<titulo_inline>[^\n]+)
    )?
    [ \t]*$
    """,
    flags=re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)

PADRAO_CAPITULO_NA_ARCA = re.compile(
    r"""
    ^[ \t]*
    cap[ií]tulo
    [ \t]+
    (?P<numero>
        [a-z]
    )
    [ \t]*$
    """,
    flags=re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)


def linha_e_numero_pagina(linha: str) -> bool:
    return bool(
        re.fullmatch(
            r"\s*\d{1,4}\s*",
            linha,
        )
    )


def linha_parece_titulo_capitulo(
    linha: str,
    max_chars: int = 120,
    max_palavras: int = 15,
) -> bool:
    """
    Heurística conservadora para título de capítulo.

    Exemplo:
        O AGREGADO
        TIO COSME
        D. GLÓRIA

    Evita classificar uma linha normal de prosa como título.
    """
    linha = linha.strip()

    if not linha:
        return False

    if linha_e_numero_pagina(linha):
        return False

    if len(linha) > max_chars:
        return False

    palavras = linha.split()

    if len(palavras) > max_palavras:
        return False

    letras = [
        c
        for c in linha
        if c.isalpha()
    ]

    if not letras:
        return False

    qtd_maiusculas = sum(
        c.isupper()
        for c in letras
    )

    proporcao_maiusculas = (
        qtd_maiusculas
        / len(letras)
    )

    return proporcao_maiusculas >= 0.80

In [8]:
def encontrar_titulo_apos_capitulo(
    linhas: list[str],
    indice_capitulo: int,
    max_palavras_fallback: int = 50,
) -> dict:

    for i in range(
        indice_capitulo + 1,
        min(indice_capitulo + 6, len(linhas)),
    ):
        linha_original = linhas[i]
        linha = linha_original.strip()

        if not linha:
            continue

        if linha_e_numero_pagina(linha):
            continue

        # Regra 1: título predominantemente em caixa alta.
        if linha_parece_titulo_capitulo(linha):
            return {
                "titulo": linha,
                "regra_titulo": "caixa_alta",
            }

        # Regra 2: primeira linha inteira após o capítulo.
        if (
            linha_original.endswith("\n")
            and len(linha.split()) <= max_palavras_fallback
        ):
            return {
                "titulo": linha,
                "regra_titulo": "proxima_linha",
            }

        return {
            "titulo": None,
            "regra_titulo": None,
        }

    return {
        "titulo": None,
        "regra_titulo": None,
    }

In [9]:
def check_obra_tem_capitulos(
    txt_path,
    titulo_obra=None,
    min_capitulos=1,
    min_mediana_chars=200,
):
    txt_path = Path(txt_path)

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    linhas = texto.splitlines(
        keepends=True
    )

    titulo_norm = (
        str(titulo_obra).strip().casefold()
        if titulo_obra
        else ""
    )

    if titulo_norm == "na arca":
        padrao_capitulo = (
            PADRAO_CAPITULO_NA_ARCA
        )
    else:
        padrao_capitulo = (
            PADRAO_CAPITULO
        )

    capitulos = []
    posicao = 0

    for indice_linha, linha_original in enumerate(
        linhas
    ):
        linha = linha_original.strip()

        match = padrao_capitulo.match(
            linha
        )

        if match:
            numero = match.group(
                "numero"
            )

            titulo_inline = (
                match.groupdict().get(
                    "titulo_inline"
                )
            )

            if titulo_inline:
                titulo = (
                    titulo_inline.strip()
                )
                regra_titulo = (
                    "inline"
                )
            else:
                resultado_titulo = (
                    encontrar_titulo_apos_capitulo(
                        linhas=linhas,
                        indice_capitulo=indice_linha,
                    )
                )

                titulo = resultado_titulo[
                    "titulo"
                ]

                regra_titulo = resultado_titulo[
                    "regra_titulo"
                ]

            capitulos.append({
                "numero": numero,
                "cabecalho": linha,
                "titulo": titulo,
                "regra_titulo": regra_titulo,
                "linha": indice_linha,
                "inicio": posicao,
            })

        posicao += len(
            linha_original
        )

    distancias = [
        atual["inicio"]
        - anterior["inicio"]
        for anterior, atual
        in zip(
            capitulos,
            capitulos[1:],
        )
    ]

    mediana_distancia = (
        median(distancias)
        if distancias
        else 0
    )

    tem_capitulos = (
        len(capitulos)
        >= min_capitulos
        and mediana_distancia
        >= min_mediana_chars
    )

    qtd_com_titulo = sum(
        capitulo["titulo"]
        is not None
        for capitulo in capitulos
    )

    return {
        "tem_capitulos": tem_capitulos,
        "qtd_capitulos_detectados": len(
            capitulos
        ),
        "qtd_capitulos_com_titulo": qtd_com_titulo,
        "pct_capitulos_com_titulo": (
            qtd_com_titulo
            / len(capitulos)
            if capitulos
            else 0
        ),
        "mediana_chars_entre_capitulos": (
            mediana_distancia
        ),
        "capitulos": capitulos,
    }

dom casmurro  

In [10]:
PROJECT_ROOT = Path("..")

dom_casmurro_path = (
    PROJECT_ROOT
    / "Dataset"
    / "bronze"
    / "Dom_Casmurro.txt"
)

resultado = check_obra_tem_capitulos(
    dom_casmurro_path,
    titulo_obra="Dom_Casmurro.txt"
)

print(
    "Tem capítulos:",
    resultado["tem_capitulos"],
)

print(
    "Capítulos:",
    resultado[
        "qtd_capitulos_detectados"
    ],
)

print(
    "Com título:",
    resultado[
        "qtd_capitulos_com_titulo"
    ],
)

print(
    "Percentual com título:",
    round(
        resultado[
            "pct_capitulos_com_titulo"
        ] * 100,
        2,
    ),
    "%",
)

print()

for capitulo in resultado[
    "capitulos"
]:
    print(
        f"{capitulo['numero']:>8} | "
        f"{capitulo['titulo']}"
    )

Tem capítulos: True
Capítulos: 145
Com título: 145
Percentual com título: 100.0 %

       I | DO TÍTULO
      II | DO LIVRO
     III | A DENÚNCIA
      IV | UM DEVER AMARÍSSIMO!
       V | O AGREGADO
      VI | TIO COSME
     VII | D. GLÓRIA
    VIII | E TEMPO
      IX | A ÓPERA
      XI | ACEITO A TEORIA
      XI | A PROMESSA
     XII | NA VARANDA
    XIII | CAPITU
     XIV | A INSCRIÇÃO
      XV | OUTRA VOZ REPENTINA
     XVI | O ADMINISTRADOR INTERINO
    XVII | OS VERMES "ELE FERE E CURA!"
   XVIII | UM PLANO
     XIX | SEM FALTA
      XX | MIL PADRE-NOSSOS E MIL AVE-MARIAS
     XXI | PRIMA JUSTINA
    XXII | SENSAÇÕES ALHEIAS
   XXIII | PRAZO DADO
    XXIV | DE MÃE E DE SERVO
    XXVI | NO PASSEIO PÚBLICO
    XXVI | AS LEIS SÃO BELAS
   XXVII | AO PORTÃO
  XXVIII | NA RUA
    XXIX | O IMPERADOR
     XXX | O SANTÍSSlMO
    XXXI | AS CURIOSIDADES DE CAPITU
   XXXII | OLHOS DE RESSACA
  XXXIII | O PENTEADO
   XXXIV | SOU HOMEM!
    XXXV | O PROTONOTÁRIO APOSTÓLICO
   XXXVI | IDÉIA SE

avaliando manualmente tem os capitulos e titulos todos, apesar de que no proprio pdf estejam errados a numeração 

O alienista

In [11]:
PROJECT_ROOT = Path("..")

O_Alienista = (
    PROJECT_ROOT
    / "Dataset"
    / "bronze"
    / "O_Alienista.txt"
)

resultado = check_obra_tem_capitulos(
    O_Alienista
)

print(
    "Tem capítulos:",
    resultado["tem_capitulos"],
)

print(
    "Capítulos:",
    resultado[
        "qtd_capitulos_detectados"
    ],
)

print(
    "Com título:",
    resultado[
        "qtd_capitulos_com_titulo"
    ],
)

print(
    "Percentual com título:",
    round(
        resultado[
            "pct_capitulos_com_titulo"
        ] * 100,
        2,
    ),
    "%",
)

print()

for capitulo in resultado[
    "capitulos"
]:
    print(
        f"{capitulo['numero']:>8} | "
        f"{capitulo['titulo']}"
    )

Tem capítulos: True
Capítulos: 13
Com título: 13
Percentual com título: 100.0 %

       I | DE COMO ITAGUAÍ GANHOU UMA CASA DE ORATES
      II | TORRENTES DE LOUCOS
     III | DEUS SABE O QUE FAZ
      IV | UMA TEORIA NOVA
       V | O TERROR
      VI | A REBELIÃO
     VII | O INESPERADO
    VIII | AS ANGÚSTIAS  DO BOTICÁRIO
      IX | DOIS LINDOS CASOS
       X | RESTAURAÇÃO
      XI | O ASSOMBRO DE ITAGUAÍ
     XII | O FINAL DO § 4º.
    XIII | PLUS  ULTRA!


avaliando manualmente está ok também titulos e numeração de capitulos, este nao tem erros na capitularização 

Memorias Postumas de bras Cubas

In [12]:
PROJECT_ROOT = Path("..")

memorias_postumas_path = (
    PROJECT_ROOT
    / "Dataset"
    / "bronze"
    / "Memórias_Póstumas_de_Brás_Cubas.txt"
)

resultado = check_obra_tem_capitulos(
    memorias_postumas_path
)

print(
    "Tem capítulos:",
    resultado["tem_capitulos"],
)

print(
    "Capítulos:",
    resultado[
        "qtd_capitulos_detectados"
    ],
)

print(
    "Com título:",
    resultado[
        "qtd_capitulos_com_titulo"
    ],
)

print(
    "Percentual com título:",
    round(
        resultado[
            "pct_capitulos_com_titulo"
        ] * 100,
        2,
    ),
    "%",
)

print()

for capitulo in resultado[
    "capitulos"
]:
    print(
        f"{capitulo['numero']:>8} | "
        f"{capitulo['titulo']}"
    )

Tem capítulos: True
Capítulos: 160
Com título: 160
Percentual com título: 100.0 %

       I | Óbito do Autor
      II | O Emplasto
     III | Genealogia
      IV | A Idéia Fixa
       V | Em Que Aparece a Orelha de Uma Senhora
      VI | Chimène, qui L Eût Dit? Rodrigue, qui L'Eût Cru?
     VII | O Delírio
    VIII | Razão Contra Sandice
      IX | Transição
       X | Naquele Dia...
      XI | O Menino é Pai do Homem
     XII | Um Episódio de 1814
    XIII | Um Salto
     XIV | O Primeiro Beijo
      XV | Marcela
     XVI | Uma Reflexão Imoral
    XVII | Do Trapézio e outras coisas
   XVIII | Visão do Corredor
     XIX | A Bordo
      XX | Bacharelo-me
     XXI | O Almocreve
    XXII | Volta ao Rio
   XXIII | Triste, mas Curto
    XXIV | Curto, mais Alegre
     XXV | Na Tijuca
    XXVI | O Autor Hesita
   XXVII | Virgília
  XXVIII | Contanto Que...
    XXIX | A Visita
     XXX | A Flor da Moita
    XXXI | A Borboleta Preta
   XXXII | Coxa de Nascença
  XXXIII | Bem-Aventurados os que 

Precisamos adicionar uma regra extra de fallback mas pra Memorias Postumas também conseguimos extrair todos os capitulos e  titulos verificados manualmente 

Bora lá testar em todas as obras agora e ver o que conseguimos: 

In [13]:
PROJECT_ROOT = Path("..")

METADATA_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "metadata"
    / "bronze"
    / "obras_machado_assis_bronze.csv"
)


df = pd.read_csv(
    METADATA_PATH
)

df_obras = df[
    (~df["is_carta"])
    & (~df["is_coletanea"])
].copy()

print(
    "Total no metadata:",
    len(df),
)

print(
    "Obras avaliadas:",
    len(df_obras),
)

Total no metadata: 257
Obras avaliadas: 246


In [14]:
resultados_capitulos = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        print(
            f"[WARN] TXT não encontrado: {txt_path}"
        )
        continue

    resultado = check_obra_tem_capitulos(
        txt_path,
        titulo_obra=row["titulo"],
    )

    resultados_capitulos.append({
        "titulo": row["titulo"],
        "bronze_text_path": row[
            "bronze_text_path"
        ],
        "tem_capitulos": resultado[
            "tem_capitulos"
        ],
        "qtd_capitulos_detectados": resultado[
            "qtd_capitulos_detectados"
        ],
        "qtd_capitulos_com_titulo": resultado[
            "qtd_capitulos_com_titulo"
        ],
        "pct_capitulos_com_titulo": resultado[
            "pct_capitulos_com_titulo"
        ],
        "mediana_chars_entre_capitulos": resultado[
            "mediana_chars_entre_capitulos"
        ],
    })


df_capitulos = pd.DataFrame(
    resultados_capitulos
)

In [15]:
df_capitulos[
    df_capitulos["tem_capitulos"]
][
    [
        "titulo",
        "qtd_capitulos_detectados",
        "qtd_capitulos_com_titulo",
        "pct_capitulos_com_titulo",
        "mediana_chars_entre_capitulos",
    ]
].sort_values(
    "qtd_capitulos_detectados",
    ascending=False,
)

,titulo,qtd_capitulos_detectados,qtd_capitulos_com_titulo,pct_capitulos_com_titulo,mediana_chars_entre_capitulos
191,Quincas Borba,200,200,1.0,1797.0
113,Memórias Póstumas de Brás Cubas,160,160,1.0,1858.0
66,Dom Casmurro,145,145,1.0,2275.0
75,Esaú e Jacó,121,121,1.0,3016.5
94,História de Quinze Dias,58,58,1.0,1281.0
124,Notas Semanais,42,42,1.0,1851.0
180,Ponto de Vista,29,29,1.0,649.5
91,Helena,28,28,1.0,10837.0
15,A Mão e a Luva,19,19,1.0,9758.0
97,Iaiá Garcia,17,17,1.0,18081.0


In [16]:
df_capitulos[
    ~df_capitulos["tem_capitulos"]
][
    [
        "titulo",
        "qtd_capitulos_detectados",
        "qtd_capitulos_com_titulo",
        "pct_capitulos_com_titulo",
        "mediana_chars_entre_capitulos",
    ]
].sort_values(
    "qtd_capitulos_detectados",
    ascending=False,
)

,titulo,qtd_capitulos_detectados,qtd_capitulos_com_titulo,pct_capitulos_com_titulo,mediana_chars_entre_capitulos
0,A Carteira,0,0,0.0,0.0
1,A Cartomante,0,0,0.0,0.0
2,A Causa Secreta,0,0,0.0,0.0
4,A Chinela Turca,0,0,0.0,0.0
5,A Desejada das Gentes,0,0,0.0,0.0
...,...,...,...,...,...
241,Viver,0,0,0.0,0.0
242,Viver!,0,0,0.0,0.0
243,Vênus! divina vênus!,0,0,0.0,0.0
244,Última receita,0,0,0.0,0.0


Opa, tivemos um pequeno edge case de "Na Arca" que teve um capitulo detectado mas parece que não teve palavras os suficiente pra passar como capitulo 

Avaliando o pdf, ele tem uma estrutura diferente, sendo numerado com letras

e.g: Capitulo A, Capitulo B e Capitulo C


Vou criar uma regra especial só pra esse case e então procurar por falsos negativos, a heuristicas disse que não tinha capitulos, mas tem 

Verificar os Falsos Negativos : 

In [17]:
df_capitulos[
    ~df_capitulos["tem_capitulos"]
][
    [
        "titulo",
        "qtd_capitulos_detectados",
        "qtd_capitulos_com_titulo",
        "pct_capitulos_com_titulo",
        "mediana_chars_entre_capitulos",
    ]
].sort_values(
    "qtd_capitulos_detectados",
    ascending=False,
)

,titulo,qtd_capitulos_detectados,qtd_capitulos_com_titulo,pct_capitulos_com_titulo,mediana_chars_entre_capitulos
0,A Carteira,0,0,0.0,0.0
1,A Cartomante,0,0,0.0,0.0
2,A Causa Secreta,0,0,0.0,0.0
4,A Chinela Turca,0,0,0.0,0.0
5,A Desejada das Gentes,0,0,0.0,0.0
...,...,...,...,...,...
241,Viver,0,0,0.0,0.0
242,Viver!,0,0,0.0,0.0
243,Vênus! divina vênus!,0,0,0.0,0.0
244,Última receita,0,0,0.0,0.0


investigar de maneira bem permissiva agora, sempre que aparecer a palavra capitulo, certamente apareceram várias, porque essa palavra pode ser usada ao longo dos textos do proprios personagens elucidando situações 

In [25]:
PADROES_INVESTIGACAO = {
    # CAPÍTULO / CAPITULO em qualquer contexto
    "palavra_capitulo": re.compile(
        r"\bcap[ií]tulo\b",
        flags=re.IGNORECASE,
    ),

    # Numerais romanos isolados como possível heading
    "romano_isolado": re.compile(
        r"(?mi)^\s*"
        r"(?:i|ii|iii|iv|v|vi|vii|viii|ix|x|"
        r"xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xix|xx)"
        r"\s*$"
    ),

    # Outras divisões estruturais
    "parte": re.compile(
        r"(?mi)^\s*parte\s+"
        r"(?:\d+|[ivxlcdm]+|[a-z])"
        r"\s*$"
    ),

    "livro": re.compile(
        r"(?mi)^\s*livro\s+"
        r"(?:\d+|[ivxlcdm]+|[a-z])"
        r"\s*$"
    ),
}

In [26]:
df_sem_capitulos = df_capitulos[
    ~df_capitulos["tem_capitulos"]
].copy()


resultados_investigacao = []

for _, row in df_sem_capitulos.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    resultado = {
        "titulo": row["titulo"],
        "bronze_text_path": row[
            "bronze_text_path"
        ],
    }

    for nome, padrao in PADROES_INVESTIGACAO.items():
        resultado[nome] = len(
            padrao.findall(texto)
        )

    resultados_investigacao.append(
        resultado
    )


df_investigacao = pd.DataFrame(
    resultados_investigacao
)

In [27]:
colunas_sinais = list(
    PADROES_INVESTIGACAO.keys()
)

df_investigacao["total_sinais"] = (
    df_investigacao[
        colunas_sinais
    ].sum(axis=1)
)

df_investigacao[
    df_investigacao["total_sinais"] > 0
].sort_values(
    "total_sinais",
    ascending=False,
)[
    [
        "titulo",
        "palavra_capitulo",
        "romano_isolado",
        "parte",
        "livro",
        "total_sinais",
    ]
]


,titulo,palavra_capitulo,romano_isolado,parte,livro,total_sinais
95,O almada,0,112,0,0,112
21,Americanas,0,56,2,0,58
61,Falenas,0,30,0,0,30
153,Ressureição,28,0,0,0,28
148,Qual dos dois,1,20,0,0,21
...,...,...,...,...,...,...
130,O Segredo do Bonzo,1,0,0,0,1
146,Páginas Críticas e Comemorativas,1,0,0,0,1
134,Onda,1,0,0,0,1
181,Um sonho e outro sonho,1,0,0,0,1


Vejam bem que avaliando os falsos negativos fiz outro achado da base, que até então fugia aos meus olhos e investigação, algumas obras são canticos e poemas, e são separados por numerais romandos e nao seguem a mesma estrutura de um conto ou livro, creio que mereçam sua propria categoria, isso passou batido por mim nos casos especiais de carta e coletanea, mas creio ser uma boa infomação, fica de achado nesse notebook, mas por hora irei ajeitar  a proposta desse de separar semanticamente capitulos e validar a heuristica 

In [32]:
def mostrar_suspeitas_capitulo(
    txt_path,
    contexto=4,
):
    texto = Path(txt_path).read_text(
        encoding="utf-8",
        errors="ignore",
    )

    linhas = texto.splitlines()

    padrao = re.compile(
        r"\bcap[ií]tulo\b|\bcap\.",
        flags=re.IGNORECASE,
    )

    for i, linha in enumerate(linhas):
        if not padrao.search(linha):
            continue

        inicio = max(
            0,
            i - contexto,
        )

        fim = min(
            len(linhas),
            i + contexto + 1,
        )

        print(
            "\n" + "=" * 80
        )

        for j in range(inicio, fim):
            marcador = (
                ">>>"
                if j == i
                else "   "
            )

            print(
                f"{marcador} "
                f"{j:05d}: "
                f"{linhas[j]}"
            )

In [31]:
suspeitos = df_investigacao[
    df_investigacao[
        "palavra_capitulo"
    ] > 0
]

for _, row in suspeitos.iterrows():

    print(
        "\n\n",
        "#" * 100,
    )

    print(
        f"TÍTULO: {row['titulo']}"
    )

    print(
        f"palavra_capitulo: {row['palavra_capitulo']} | "
        f"romano_isolado: {row['romano_isolado']} | "
        f"parte: {row['parte']} | "
        f"livro: {row['livro']} | "
        f"total_sinais: {row['total_sinais']}"
    )

    print(
        f"ARQUIVO: {row['bronze_text_path']}"
    )

    print(
        "-" * 100
    )

    mostrar_suspeitas_capitulo(
        PROJECT_ROOT
        / row["bronze_text_path"]
    )



 ####################################################################################################
TÍTULO: A mulher Pálida
palavra_capitulo: 1 | romano_isolado: 4 | parte: 0 | livro: 0 | total_sinais: 5
ARQUIVO: Dataset/bronze/A_mulher_Pálida.txt
----------------------------------------------------------------------------------------------------

    00306: delicadeza e de um amor-próprio extraordinários. Essa era a outra feição principal dele, a
    00307: que me dá esta novelita; porque se tal não fora... Mas eu não quero usurpar a ação do
>>> 00308: capítulo seguinte.
    00309: IV
    00310: 


 ####################################################################################################
TÍTULO: A Pianista
palavra_capitulo: 1 | romano_isolado: 0 | parte: 0 | livro: 0 | total_sinais: 1
ARQUIVO: Dataset/bronze/A_Pianista.txt
----------------------------------------------------------------------------------------------------

    00395: não foi o algoz, foi o pai.
    0039